# 🎮📊 Análise pedagógica de sessões de criação de jogos

**Paula Peclat** · [APedê Digital](https://paulapeclat.com.br) · dados **sintéticos** (nenhuma criança real)

Três perguntas pedagógicas guiam esta análise:

1. **Engajamento** — quanto tempo as crianças permanecem ativas em cada ferramenta?
2. **Progressão** — a conclusão de etapas cresce com a idade e com a prática (nº da sessão)?
3. **Autonomia** — em quais ferramentas as crianças mais pedem ajuda, e o que isso diz sobre o degrau de dificuldade?

> ⚖️ Antes de usar com dados reais: LGPD art. 14 (consentimento parental), anonimização na coleta e finalidade pedagógica explícita. Veja o README.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# paleta paulapeclat.com.br
CORES = ["#22C3D9", "#F23D91", "#ECBE40", "#F2A2C0"]
plt.rcParams["figure.figsize"] = (9, 4.5)

df = pd.read_csv("dados/sessoes_exemplo.csv")
print(f"{len(df)} sessões · {df.aluno.nunique()} alunos · {df.ferramenta.nunique()} ferramentas")
df.head()

## 1. Engajamento por ferramenta

Minutos de atividade não medem aprendizagem — mas quedas bruscas de engajamento costumam sinalizar que o degrau de dificuldade ficou alto demais (ou baixo demais).

In [ ]:
resumo = (df.groupby("ferramenta")
            .agg(sessoes=("aluno", "count"),
                 minutos_medios=("minutos", "mean"),
                 etapas_medias=("etapas_concluidas", "mean"),
                 pct_pediu_ajuda=("pediu_ajuda", "mean"))
            .round(2)
            .sort_values("minutos_medios"))
resumo["pct_pediu_ajuda"] = (resumo["pct_pediu_ajuda"] * 100).round(0)
resumo

In [ ]:
fig, ax = plt.subplots()
resumo["minutos_medios"].plot.barh(ax=ax, color=CORES)
ax.set_xlabel("minutos médios por sessão")
ax.set_ylabel("")
ax.set_title("Engajamento médio por ferramenta")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 2. Progressão: prática e idade

Se a mediação está funcionando, a conclusão de etapas deve subir da 1ª para a 3ª sessão — o efeito da **prática** — em todas as idades, e não apenas entre os mais velhos.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))

por_sessao = df.groupby("sessao_num")["etapas_concluidas"].mean()
a1.plot(por_sessao.index, por_sessao.values, marker="o", color="#F23D91", linewidth=2)
a1.set_xticks([1, 2, 3])
a1.set_xlabel("nº da sessão")
a1.set_ylabel("etapas concluídas (média)")
a1.set_title("Efeito da prática")
a1.spines[["top", "right"]].set_visible(False)

por_idade = df.groupby("idade")["etapas_concluidas"].mean()
a2.bar(por_idade.index, por_idade.values, color="#22C3D9")
a2.set_xlabel("idade")
a2.set_title("Etapas concluídas por idade")
a2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Autonomia: quem pede ajuda, onde

Pedir ajuda **não é fracasso** — é interação legítima. Mas se uma ferramenta concentra muito mais pedidos que as outras, o roteiro daquela ferramenta provavelmente pula degraus.

In [ ]:
ajuda = df.groupby("ferramenta")["pediu_ajuda"].mean().sort_values() * 100

fig, ax = plt.subplots()
ax.barh(ajuda.index, ajuda.values, color=CORES)
ax.set_xlabel("% de sessões com pedido de ajuda")
ax.set_title("Pedidos de ajuda por ferramenta")
ax.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(ajuda.values):
    ax.text(v + 1, i, f"{v:.0f}%", va="center")
plt.tight_layout()
plt.show()

## Leitura pedagógica (não estatística)

Com dados no padrão deste exemplo, três decisões de aula se sustentam:

1. **A escada de ferramentas funciona** — engajamento e pedidos de ajuda crescem juntos do Code.org ao Roblox Studio, o que é esperado quando o degrau de abstração sobe. O ponto de atenção seria o inverso: ajuda alta com engajamento baixo.
2. **Prática importa mais que idade** — se a curva por sessão sobe de forma consistente, vale garantir 3+ sessões por ferramenta antes de avançar, em vez de acelerar os mais velhos.
3. **Ajuda concentrada = roteiro a revisar** — a ferramenta com mais pedidos de ajuda merece um passo intermediário no roteiro, não crianças "mais preparadas".

### Limites

- Dados sintéticos: os padrões aqui são ilustrativos do **método**, não achados empíricos.
- `minutos` mede permanência, não profundidade; triangule com observação e com as produções das crianças.
- n pequeno por célula (idade × ferramenta): leia tendências, não teste hipóteses.

### Próximos passos

- Substituir o CSV por dados reais **anonimizados e consentidos** (LGPD art. 14)
- Adicionar as *produções* (jogos criados) como dado qualitativo pareado
- Comparar turmas síncronas × assíncronas